In [1]:
from pyspark.sql import SparkSession

_PACKAGES = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2",
])


def createSpark():
    spark = (SparkSession.builder
             .appName("dev")
             .master("local[*]")
             .config("spark.jars.packages", _PACKAGES)
             # S3A / MinIO
             .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
             .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
             .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
             .config("spark.hadoop.fs.s3a.path.style.access", "true")
             .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
             .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
             # Iceberg extensions
             .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
             # Iceberg catalog "lake"
             .config("spark.sql.catalog.lake", "org.apache.iceberg.spark.SparkCatalog")
             .config("spark.sql.catalog.lake.type", "hadoop")
             .config("spark.sql.catalog.lake.warehouse", "s3a://iceberg-lakehouse/warehouse")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    return spark


spark = createSpark()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3e63644f-76d0-4051-b2f2-a045573ad7c3;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.

In [2]:
create_prices = '''CREATE TABLE IF NOT EXISTS lake.silver.prices (
    -- Идентификация
    coin_id                          STRING  NOT NULL,
    symbol                           STRING  NOT NULL,
    name                             STRING,
    image_url                        STRING,

    -- Цена и капитализация
    current_price_usd                DECIMAL(28, 12),
    market_cap_usd                   DECIMAL(28, 4),
    market_cap_rank                  INT,
    fully_diluted_valuation_usd      DECIMAL(28, 4),
    total_volume_usd                 DECIMAL(28, 4),

    -- Диапазон за 24ч
    high_24h_usd                     DECIMAL(28, 12),
    low_24h_usd                      DECIMAL(28, 12),

    -- Изменения
    price_change_24h_usd             DECIMAL(28, 12),
    price_change_percentage_24h      DOUBLE,
    market_cap_change_24h_usd        DECIMAL(28, 4),
    market_cap_change_percentage_24h DOUBLE,

    -- Supply
    circulating_supply               DECIMAL(38, 8),
    total_supply                     DECIMAL(38, 8),
    max_supply                       DECIMAL(38, 8),

    -- ATH / ATL
    ath_usd                          DECIMAL(28, 12),
    ath_change_percentage            DOUBLE,
    ath_date                         TIMESTAMP,
    atl_usd                          DECIMAL(28, 12),
    atl_change_percentage            DOUBLE,
    atl_date                         TIMESTAMP,

    -- Время снимка
    last_updated                     TIMESTAMP,
    snapshot_ts                      TIMESTAMP NOT NULL,
    ingestion_ts                     TIMESTAMP NOT NULL
)
USING iceberg
PARTITIONED BY (days(snapshot_ts), bucket(4, coin_id))
TBLPROPERTIES (
    'write.target-file-size-bytes'               = '134217728',
    'write.distribution-mode'                    = 'hash',
    'write.parquet.compression-codec'            = 'snappy',
    'format-version'                             = '2',
    'history.expire.max-snapshot-age-ms'         = '604800000',
    'write.metadata.delete-after-commit.enabled' = 'true',
    'write.metadata.previous-versions-max'       = '20'
);'''

spark.sql(create_prices)

DataFrame[]

In [6]:
create_tickers = '''
CREATE TABLE IF NOT EXISTS lake.silver.tickers (
    -- Идентификация
    symbol              STRING    NOT NULL,
    
    -- Тип события (snapshot/delta) и cross-sequence для упорядочивания
    event_type          STRING    NOT NULL,
    cross_seq           BIGINT,
    
    -- Время события (из Bybit) и системные метки
    event_ts            TIMESTAMP NOT NULL,
    ingestion_ts        TIMESTAMP NOT NULL,
    
    -- Цены
    last_price          DECIMAL(28, 12),
    high_price_24h      DECIMAL(28, 12),
    low_price_24h       DECIMAL(28, 12),
    prev_price_24h      DECIMAL(28, 12),
    usd_index_price     DECIMAL(28, 12),
    
    -- Объёмы
    volume_24h          DECIMAL(38, 8),
    turnover_24h        DECIMAL(38, 8),
    
    -- Изменение за 24ч
    price_24h_pcnt      DOUBLE
)
USING iceberg
PARTITIONED BY (hours(event_ts), bucket(8, symbol))
TBLPROPERTIES (
    'write.target-file-size-bytes'               = '134217728',
    'write.distribution-mode'                    = 'hash',
    'write.parquet.compression-codec'            = 'snappy',
    'format-version'                             = '2',
    'history.expire.max-snapshot-age-ms'         = '604800000',
    'write.metadata.delete-after-commit.enabled' = 'true',
    'write.metadata.previous-versions-max'       = '20'
);'''

spark.sql(create_tickers)

DataFrame[]

In [10]:
spark.sql("DESCRIBE lake.silver.tickers;").show(truncate=False)

+---------------+-----------------+-------+
|col_name       |data_type        |comment|
+---------------+-----------------+-------+
|symbol         |string           |NULL   |
|event_type     |string           |NULL   |
|cross_seq      |bigint           |NULL   |
|event_ts       |timestamp        |NULL   |
|ingestion_ts   |timestamp        |NULL   |
|last_price     |decimal(28,12)   |NULL   |
|high_price_24h |decimal(28,12)   |NULL   |
|low_price_24h  |decimal(28,12)   |NULL   |
|prev_price_24h |decimal(28,12)   |NULL   |
|usd_index_price|decimal(28,12)   |NULL   |
|volume_24h     |decimal(38,8)    |NULL   |
|turnover_24h   |decimal(38,8)    |NULL   |
|price_24h_pcnt |double           |NULL   |
|               |                 |       |
|# Partitioning |                 |       |
|Part 0         |hours(event_ts)  |       |
|Part 1         |bucket(8, symbol)|       |
+---------------+-----------------+-------+



In [12]:
spark.sql('ALTER TABLE lake.silver.tickers ALTER COLUMN symbol DROP NOT NULL;').show()
spark.sql('ALTER TABLE lake.silver.tickers ALTER COLUMN event_type DROP NOT NULL;').show()
spark.sql('ALTER TABLE lake.silver.tickers ALTER COLUMN event_ts DROP NOT NULL;').show()
spark.sql('ALTER TABLE lake.silver.tickers ALTER COLUMN ingestion_ts DROP NOT NULL;').show()

++
||
++
++

++
||
++
++

++
||
++
++

++
||
++
++

